<img src="https://raw.githubusercontent.com/tancao1128/ams261-labs/main/assets/lab15-banner.png" alt="AMS 261 Lab 15 · Slice Probe: the surface z = −x²/2 − y² + 25/8 cut by the planes y = 1 and x = 1/2 at the point (1/2, 1, 2)" width="100%">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tancao1128/ams261-labs/blob/main/labs/AMS261-Lecture15.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/tancao1128/ams261-labs/main?labpath=labs%2FAMS261-Lecture15.ipynb)
· [Download this notebook (.ipynb)](https://raw.githubusercontent.com/tancao1128/ams261-labs/main/labs/AMS261-Lecture15.ipynb)

## 0 · Mission briefing

A probe stands on the surface $z=f(x,y)$ with
$$
f(x,y)=-\frac{x^2}{2}-y^2+\frac{25}{8}.
$$
It stands at the point $\left(\tfrac12,\,1,\,2\right)$. This is Example 2 on the Lecture 15 page.
The probe can measure the surface only along two vertical planes through its position.
The plane $y=1$ cuts the surface in one curve. The plane $x=\tfrac12$ cuts it in another.
In this lab we measure the slopes of these two slices. They are the partial derivatives $f_x$ and $f_y$.

- **Time:** about 30 minutes.
- **When:** after class, before Lecture 16.
- **Before you start:** read the page sections *Geometric motivation*, *Formal definition* and *Using the definition directly*, and Examples 2, 3 and 9 on the Lecture 15 page.
- **Section numbers.** Sections are numbered 0–14. The Lecture 15 page has no section numbers. So we name its section titles and its Example numbers.

**Lab objectives.** By the end of this lab you can

1. compute a partial derivative from the limit definition and check it against the rule;
2. read $f_x(a,b)$ and $f_y(a,b)$ as the slopes of two slices of the surface;
3. explain why two zero slopes do not make a surface flat, and check Clairaut's theorem on Example 9.

**Two slices.** Keep them apart.

- The **$x$-slice** is the curve $z=f(x,b)$. It lies in the plane $y=b$. Its slope at $x=a$ is $f_x(a,b)$.
- The **$y$-slice** is the curve $z=f(a,y)$. It lies in the plane $x=a$. Its slope at $y=b$ is $f_y(a,b)$.

**How to work.** Run the cells from top to bottom with Shift+Enter. A cell marked ✏️ asks you to type something first.

> **Not graded. Practice only.**

In [ ]:
# SETUP: run this cell first. It needs no local files.
import base64, json, math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3D projection)
from IPython.display import HTML, Markdown, display

try:
    from ipywidgets import (FloatSlider, Button, Label, HBox, VBox, Layout,
                            HTML as WidgetHTML, interactive_output)
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

x, y = sp.symbols('x y', real=True)

# Colour tokens. Each colour has one meaning in every plot of this lab.
INK = '#1E293B'        # text and axes
PRIMARY = '#2563EB'    # a second quantity drawn next to a slice (for example a slope curve)
NEGATIVE = '#DC2626'   # a downward-curving (concave down) slice
TANGENT = '#059669'    # tangent lines; their slopes are the partial derivatives
PROBE = '#D97706'      # the probe point (a, b, f(a, b))
CONTOUR = '#94A3B8'    # the surface mesh and reference lines
SLICE = '#7C3AED'      # slice curves of the surface

# Console tokens. They are used only for the frame around each plot.
HUD_NAVY = '#0A0E14'   # top and bottom bars
HUD_PANEL = '#0F1A2B'  # the outer frame
CYAN = '#22D3EE'       # frame text and corner brackets
SCREEN = '#F8FAFC'     # the light screen behind every panel

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})


def show_math(latex):
    # Display one formula.
    display(Markdown(f"$${latex}$$"))


def show_text(text):
    # Display formatted text.
    display(Markdown(text))


def fmt(v, digits=2):
    # Format a number; never print -0.00.
    v = float(v)
    if abs(v) < 0.5 * 10 ** (-digits):
        v = 0.0
    return f"{v:.{digits}f}"


def as_number(v):
    # Accept a number or a string such as "-1/2" or "100*sqrt(3)".
    return float(sp.sympify(v))


def check_answer(given, target, tol=5e-3):
    # Compare an answer with a target. Returns True or False; never raises.
    try:
        if given is None:
            return False
        if isinstance(target, bool):
            return given is target
        if isinstance(target, str):
            return str(given).strip().lower() == target
        if isinstance(target, (list, tuple)):
            return (len(given) == len(target)
                    and all(check_answer(a, b, tol) for a, b in zip(given, target)))
        return math.isclose(as_number(given), float(target), rel_tol=1e-3, abs_tol=tol)
    except Exception:
        return False


def hud_frame(fig, label, footer='SLICE PROBE · f = −x²/2 − y² + 25/8 · P = (1/2, 1)'):
    # Put a finished figure inside a navy console frame.
    # All panels, labels and legends sit on one light screen, so axis text stays dark on light.
    side, top, foot, gap, pad = 0.30, 0.40, 0.30, 0.10, 0.12   # sizes in inches
    w, h = fig.get_size_inches()
    W, H = w + 2 * (side + pad), h + top + foot + 2 * (gap + pad)
    fig.set_size_inches(W, H)
    fig.patch.set_facecolor(HUD_PANEL)

    # Dark ink on the light screen.
    for ax in fig.axes:
        ax.set_facecolor(SCREEN)
        if hasattr(ax, 'zaxis'):      # a 3D panel: light panes, dark z-label
            for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
                axis.set_pane_color((0.973, 0.980, 0.988, 1.0))
            ax.zaxis.label.set_color(INK)
        ax.tick_params(colors=INK, labelcolor=INK)
        ax.title.set_color(INK)
        ax.xaxis.label.set_color(INK)
        ax.yaxis.label.set_color(INK)
        for spine in ax.spines.values():
            spine.set_color('#64748B')
        if ax.get_legend() is not None:
            for t in ax.get_legend().get_texts():
                t.set_color(INK)

    # The screen, and the area inside it where the panels go.
    sx0, sy0, sx1, sy1 = side, foot + gap, W - side, H - top - gap   # in inches
    box = [sx0 + pad, sy0 + pad, sx1 - pad, sy1 - pad]
    for _ in range(3):
        fig.tight_layout(rect=(box[0] / W, box[1] / H, box[2] / W, box[3] / H))
        bb = fig.get_tightbbox(fig.canvas.get_renderer())   # everything drawn so far, in inches
        over = [sx0 + pad - bb.x0, sy0 + pad - bb.y0, bb.x1 - sx1 + pad, bb.y1 - sy1 + pad]
        if max(over) < 0.01:
            break   # labels and legends all fit on the screen
        box = [box[0] + max(over[0], 0), box[1] + max(over[1], 0),
               box[2] - max(over[2], 0), box[3] - max(over[3], 0)]
    fig.add_artist(Rectangle((sx0 / W, sy0 / H), (sx1 - sx0) / W, (sy1 - sy0) / H,
                             transform=fig.transFigure, facecolor=SCREEN,
                             edgecolor='#334155', lw=1, zorder=-10))

    # A faint grid on the screen, about 24 px apart.
    step = 0.24
    segs = [[(a / W, sy0 / H), (a / W, sy1 / H)] for a in np.arange(sx0 + step, sx1, step)]
    segs += [[(sx0 / W, b / H), (sx1 / W, b / H)] for b in np.arange(sy0 + step, sy1, step)]
    fig.add_artist(LineCollection(segs, colors='#E2E8F0', linewidths=0.6,
                                  transform=fig.transFigure, zorder=-9))

    # Top bar with the section label, and a small footer bar.
    for y_bar, height in [(H - top, top), (0, foot)]:
        fig.add_artist(Rectangle((0, y_bar / H), 1, height / H, transform=fig.transFigure,
                                 facecolor=HUD_NAVY, edgecolor='none', zorder=-8))
    fig.add_artist(Line2D([0, 1], [(H - top) / H] * 2, transform=fig.transFigure, color=CYAN, lw=1.5))
    fig.add_artist(Line2D([0, 1], [foot / H] * 2, transform=fig.transFigure, color=CYAN, lw=0.8, alpha=0.6))
    mono = dict(family='monospace', transform=fig.transFigure, va='center')
    fig.text(side / W, 1 - top / (2 * H), 'AMS 261 · M15 · ' + label, color=CYAN,
             fontsize=10, fontweight='bold', ha='left', **mono)
    fig.text(1 - side / W, 1 - top / (2 * H), '● PROBE LINK · LIVE', color=CYAN,
             fontsize=8, ha='right', **mono)
    fig.text(side / W, foot / (2 * H), footer, color=CONTOUR, fontsize=8, ha='left', **mono)
    fig.text(1 - side / W, foot / (2 * H), 'MISSION 15', color=CONTOUR, fontsize=8, ha='right', **mono)

    # Cyan corner brackets just outside the screen.
    off, arm = 0.07, 0.24
    for cx, cy, dx, dy in [(sx0 - off, sy0 - off, 1, 1), (sx1 + off, sy0 - off, -1, 1),
                           (sx0 - off, sy1 + off, 1, -1), (sx1 + off, sy1 + off, -1, -1)]:
        fig.add_artist(Line2D([(cx + dx * arm) / W, cx / W, cx / W],
                              [cy / H, cy / H, (cy + dy * arm) / H],
                              transform=fig.transFigure, color=CYAN, lw=2))
    return fig


CARD = ('max-width:560px;margin-top:6px;border:1px solid #0F1A2B;border-radius:10px;overflow:hidden;'
        'background:#F8FAFC;color:#1E293B;font-family:system-ui,-apple-system,Segoe UI,sans-serif;')
CARD_HEAD = ('background:#0F1A2B;color:#22D3EE;padding:6px 14px;border-bottom:2px solid #22D3EE;'
             'font-family:Menlo,Consolas,monospace;letter-spacing:2px;font-size:12px;')


def check_card(title, rows, summary):
    # A small result card. Each row is (item, mark, words), for example ('f_x_P', '✅', 'correct').
    body = ''.join(
        f'<div style="padding:3px 0;color:#1E293B;">'
        f'<span style="display:inline-block;min-width:260px;padding-right:12px;font-family:Menlo,Consolas,monospace;">{item}</span>'
        f'{mark} {words}</div>' for item, mark, words in rows)
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">{title}</div>'
                 f'<div style="padding:8px 14px;color:#1E293B;">{body}'
                 f'<div style="margin-top:6px;font-weight:600;color:#1E293B;">{summary}</div></div></div>'))


def mission_complete_card():
    # Shown once the mission log and the retrieval check are both correct.
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">✅ MISSION 15 COMPLETE</div>'
                 '<div style="padding:8px 14px;line-height:1.5;color:#1E293B;">'
                 'Your mission log and your retrieval check are both correct.<br>'
                 '<b>Prediction for Lecture 16.</b> Both partials exist at a point. '
                 'Is the tangent plane built from them always a good approximation?<br>'
                 'Write your guess in one sentence. Lecture 16 will test it.</div></div>'))


print('Setup complete. Widgets available:', HAS_WIDGETS)

---
## 1 · PREDICT · Slice sweep

🔮 **STAGE 1/7 · PREDICT** ▰▱▱▱▱▱▱

The probe stands at $P=\left(\tfrac12,1\right)$ on the surface $z=-\tfrac{x^2}{2}-y^2+\tfrac{25}{8}$, at height $2$.
The highest point of this surface is above the origin.

- The $x$-slice through $P$ is the curve $z=f(x,1)$ in the plane $y=1$.
- The $y$-slice through $P$ is the curve $z=f\left(\tfrac12,y\right)$ in the plane $x=\tfrac12$.

**Do not compute yet.** Make two guesses.

1. Walking in the $+x$ direction from $P$, does the probe go uphill or downhill? And in the $+y$ direction?
2. Which slice is steeper at $P$: the $x$-slice or the $y$-slice?

A rough guess is fine. You will check it in Section 4.

In [ ]:
# ✏️ PREDICT. Type your two guesses, then run this cell. Do not compute first.
my_direction_guess = None   # for example 'x: uphill, y: downhill'
my_steeper_guess = None     # 'x' or 'y'

if my_direction_guess is None or my_steeper_guess is None:
    print('Prediction not logged yet. Type two guesses above and run this cell again.')
else:
    print(f'Prediction logged: {my_direction_guess}; steeper slice: {my_steeper_guess}.')
    print('You will check it in Section 4.')

In [ ]:
# The surface of Example 2 on the Lecture 15 page.
f_expr = -x**2/2 - y**2 + sp.Rational(25, 8)
P = (sp.Rational(1, 2), sp.Integer(1))

f_at_P = f_expr.subs({x: P[0], y: P[1]})
print(f'f(P) = {f_at_P} at P = (1/2, 1). So the probe stands at (1/2, 1, {f_at_P}).')

---
## 2 · EXPLORE · Definition core

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

The page section *Formal definition* gives the two partial derivatives at $(a,b)$:

$$
f_x(a,b)=\lim_{h\to 0}\frac{f(a+h,b)-f(a,b)}{h},\qquad
f_y(a,b)=\lim_{h\to 0}\frac{f(a,b+h)-f(a,b)}{h}.
$$

Observe that only one coordinate moves in each quotient. The other one is frozen.
The engine `partial_limit` below evaluates these limits with sympy.

**Page example (*Using the definition directly*).** Let $f(x,y)=x^2+3xy$. Compute $f_x(1,2)$ from the definition.

In [ ]:
h = sp.symbols('h')


def difference_quotient(f, var, point):
    # The difference quotient of the definition: only the coordinate `var` moves by h.
    a, b = point
    if var == x:
        moved = f.subs({x: a + h, y: b}, simultaneous=True)
    elif var == y:
        moved = f.subs({x: a, y: b + h}, simultaneous=True)
    else:
        raise ValueError('var must be x or y')
    return (moved - f.subs({x: a, y: b}, simultaneous=True)) / h


def partial_limit(f, var, point):
    # The partial derivative at `point` from the limit definition.
    return sp.limit(difference_quotient(f, var, point), h, 0)


g_def = x**2 + 3*x*y
q = sp.simplify(difference_quotient(g_def, x, (1, 2)))
show_math(r'\frac{f(1+h,\,2)-f(1,2)}{h} = ' + sp.latex(q))
show_math(r'f_x(1,2)\ \text{(limit)} = ' + sp.latex(partial_limit(g_def, x, (1, 2))))

**Step 1 · Difference quotient:** $\dfrac{f(1+h,2)-f(1,2)}{h}$.
- Move only $x$: replace $(x,y)$ by $(1+h,2)$. Keep $y=2$ frozen.
- Simplify, as the cell above does. The quotient is $8+h$.

**Step 2 · Limit as $h\to0$:** $f_x(1,2)=\lim_{h\to0}(8+h)$.
- Let $h\to0$. The quotient tends to $8$.
- Check with the rule, as the page does. Note that the $3xy$ term contributed $3y=6$, exactly as if $y$ were the constant $2$.

**Answer.** $f_x(1,2)=8$. The limit definition and the rule give the same slope.

**Now point the engine at the probe.** The cell below writes both quotients for Example 2 at $P$.
Then it prints a few values of each quotient as $h$ shrinks.

In [ ]:
qx = sp.simplify(difference_quotient(f_expr, x, P))
qy = sp.simplify(difference_quotient(f_expr, y, P))
show_math(r'\frac{f(\tfrac12+h,\,1)-f(\tfrac12,1)}{h} = ' + sp.latex(qx)
          + r',\qquad \frac{f(\tfrac12,\,1+h)-f(\tfrac12,1)}{h} = ' + sp.latex(qy))

print(f"{'h':>8}{'x-quotient':>14}{'y-quotient':>14}")
for hv in [sp.Rational(1, 10), sp.Rational(1, 100), sp.Rational(1, 1000), -sp.Rational(1, 1000)]:
    print(f"{float(hv):>8}{fmt(qx.subs(h, hv), 4):>14}{fmt(qy.subs(h, hv), 4):>14}")

Read the two columns. Each one settles down as $h\to0$. Write the two limits on paper before you go on.

---
## 3 · EXPLORE · Slicing the surface

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

The page sections *Constructing the partial derivative with respect to $x$* and *with respect to $y$* explain the picture.
The plane $y=b$ cuts the surface in the curve $z=f(x,b)$. Its slope at $x=a$ is $f_x(a,b)$.
The plane $x=a$ cuts the surface in the curve $z=f(a,y)$. Its slope at $y=b$ is $f_y(a,b)$.

The cell below computes $f_x$ and $f_y$ in two ways. The first way is the rule (*treat the other variable as a constant*). The second way is the limit engine of Section 2.
Then it **checks** that the two ways agree at $P$.

In [ ]:
fx_expr = sp.diff(f_expr, x)     # rule: treat y as a constant
fy_expr = sp.diff(f_expr, y)     # rule: treat x as a constant
show_math(r'f_x(x,y) = ' + sp.latex(fx_expr) + r',\qquad f_y(x,y) = ' + sp.latex(fy_expr))

fx_P_rule = fx_expr.subs({x: P[0], y: P[1]})
fy_P_rule = fy_expr.subs({x: P[0], y: P[1]})
fx_P_limit = partial_limit(f_expr, x, P)
fy_P_limit = partial_limit(f_expr, y, P)
print('Rule and limit agree for f_x at P:', sp.simplify(fx_P_rule - fx_P_limit) == 0)
print('Rule and limit agree for f_y at P:', sp.simplify(fy_P_rule - fy_P_limit) == 0)

# Numerical copies for the plots below.
f_num = sp.lambdify((x, y), f_expr, 'numpy')
fx_num = sp.lambdify((x, y), fx_expr, 'numpy')
fy_num = sp.lambdify((x, y), fy_expr, 'numpy')

---
## 4 · EXPLAIN · Slice probe

🧠 **STAGE 3/7 · EXPLAIN** ▰▰▰▱▱▱▱

The probe console has two sliders, $a$ and $b$. They move the probe to the point $(a,b,f(a,b))$.
Three screens show what the probe sees.

- Left: the surface, the probe, both slices and both tangent lines.
- Middle: the $x$-slice $z=f(x,b)$ with its tangent line at $x=a$. The slope of this line is $f_x(a,b)$.
- Right: the $y$-slice $z=f(a,y)$ with its tangent line at $y=b$. The slope of this line is $f_y(a,b)$.

**Your task.**

- Press **P** to put the probe at $P=\left(\tfrac12,1\right)$. Read $f_x(P)$ and $f_y(P)$ from the readout.
- Move only $b$. Watch the slope of the $x$-slice. Does it change?
- Find the point $(a,b)$ where **both** slices are level, that is, both slopes are $0$.

If the sliders do not appear, use the static cell after them. Change `a_static` and `b_static` there and run it again.

In [ ]:
grid = np.linspace(-2, 2, 41)
GX, GY = np.meshgrid(grid, grid)
GZ = f_num(GX, GY)
Z_LIM = (-3.5, 4.0)


def probe_readings(a, b):
    # Height of the surface and the two slice slopes at (a, b).
    return float(f_num(a, b)), float(fx_num(a, b)), float(fy_num(a, b))


def plot_probe(a=0.5, b=1.0):
    z0, sx, sy = probe_readings(a, b)
    fig = plt.figure(figsize=(14, 5.6))
    ax3 = fig.add_subplot(1, 3, 1, projection='3d')
    axx = fig.add_subplot(1, 3, 2)
    axy = fig.add_subplot(1, 3, 3)
    t = np.linspace(-2, 2, 200)
    s = np.linspace(-0.9, 0.9, 2)

    # Left panel: the surface with both slices and both tangent lines.
    ax3.plot_wireframe(GX, GY, GZ, rstride=3, cstride=3, color=CONTOUR, linewidth=0.5)
    ax3.plot(t, np.full_like(t, b), f_num(t, b), color=SLICE, lw=2.6)
    ax3.plot(np.full_like(t, a), t, f_num(a, t), color=SLICE, lw=2.6, ls='--')
    ax3.plot(a + s, np.full_like(s, b), z0 + sx*s, color=TANGENT, lw=3)
    ax3.plot(np.full_like(s, a), b + s, z0 + sy*s, color=TANGENT, lw=3, ls='--')
    ax3.scatter([a], [b], [z0], color=PROBE, s=60, depthshade=False)
    ax3.view_init(elev=24, azim=-112)
    ax3.set_xlim(-2, 2); ax3.set_ylim(-2, 2); ax3.set_zlim(*Z_LIM)
    ax3.set_xticks([-2, 0, 2]); ax3.set_yticks([-2, 0, 2]); ax3.set_zticks([-2, 0, 2, 4])
    ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')
    ax3.set_title('Surface z = f(x, y) and the probe')
    ax3.legend(handles=[
        Line2D([0], [0], color=CONTOUR, lw=1, label='grey mesh: the surface'),
        Line2D([0], [0], color=SLICE, lw=2.4, label='purple solid: x-slice (plane y = b)'),
        Line2D([0], [0], color=SLICE, lw=2.4, ls='--', label='purple dashed: y-slice (plane x = a)'),
        Line2D([0], [0], color=TANGENT, lw=2.6, label=r'green solid: tangent, slope $f_x$'),
        Line2D([0], [0], color=TANGENT, lw=2.6, ls='--', label=r'green dashed: tangent, slope $f_y$'),
        Line2D([0], [0], color=PROBE, marker='o', lw=0, ms=8, label='amber dot: the probe'),
    ], loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=7.5, frameon=False)

    # Middle panel: the x-slice in the plane y = b.
    axx.plot(t, f_num(t, b), color=SLICE, lw=2.2, label=f'purple: x-slice z = f(x, {b:.2f})')
    axx.plot(a + s*1.5, z0 + sx*s*1.5, color=TANGENT, lw=2.4,
             label=r'green: tangent line, slope $f_x(a,b)$')
    axx.plot([a], [z0], 'o', color=PROBE, ms=9, label='amber dot: the probe')
    axx.set_xlim(-2, 2); axx.set_ylim(*Z_LIM)
    axx.set_xlabel('x'); axx.set_ylabel('z')
    axx.set_title(rf'x-slice (plane y = {b:.2f}):  $f_x$ = {fmt(sx)}')
    axx.grid(alpha=0.3)
    axx.legend(loc='lower center', fontsize=8.5)

    # Right panel: the y-slice in the plane x = a.
    axy.plot(t, f_num(a, t), color=SLICE, lw=2.2, ls='--', label=f'purple dashed: y-slice z = f({a:.2f}, y)')
    axy.plot(b + s*1.5, z0 + sy*s*1.5, color=TANGENT, lw=2.4, ls='--',
             label=r'green dashed: tangent line, slope $f_y(a,b)$')
    axy.plot([b], [z0], 'o', color=PROBE, ms=9, label='amber dot: the probe')
    axy.set_xlim(-2, 2); axy.set_ylim(*Z_LIM)
    axy.set_xlabel('y'); axy.set_ylabel('z')
    axy.set_title(rf'y-slice (plane x = {a:.2f}):  $f_y$ = {fmt(sy)}')
    axy.grid(alpha=0.3)
    axy.legend(loc='lower center', fontsize=8.5)

    hud_frame(fig, 'SEC 4 · SLICE PROBE')
    display(fig)      # display, then close: works the same with or without the sliders
    plt.close(fig)


if HAS_WIDGETS:
    # The probe console: two full-width sliders, one preset and a text readout.
    a_slider = FloatSlider(min=-2, max=2, step=0.05, value=-1.5, description='a (x of probe)',
                           readout_format='.2f', continuous_update=False,
                           layout=Layout(width='100%'),
                           style={'handle_color': CYAN, 'description_width': 'initial'})
    b_slider = FloatSlider(min=-2, max=2, step=0.05, value=-1.0, description='b (y of probe)',
                           readout_format='.2f', continuous_update=False,
                           layout=Layout(width='100%'),
                           style={'handle_color': CYAN, 'description_width': 'initial'})
    # The only preset is the mission point P from the briefing.
    # The level point is what the mission log asks you to find, so you find it with the sliders.
    p_button = Button(description='P', tooltip='Move the probe to P = (1/2, 1)', layout=Layout(width='auto'))

    def go_to_P(_button):
        a_slider.value, b_slider.value = 0.5, 1.0

    p_button.on_click(go_to_P)
    readout = WidgetHTML()

    def update_readout(change=None):
        z0, sx, sy = probe_readings(a_slider.value, b_slider.value)
        readout.value = (f'<span style="font-family:Menlo,Consolas,monospace;">READOUT · '
                         f'(a, b) = ({a_slider.value:.2f}, {b_slider.value:.2f}) · '
                         f'height f = {fmt(z0, 3)} · slope f<sub>x</sub> = {fmt(sx, 3)} · '
                         f'slope f<sub>y</sub> = {fmt(sy, 3)}</span>')

    a_slider.observe(update_readout, names='value')
    b_slider.observe(update_readout, names='value')
    update_readout()
    plot_screen = interactive_output(plot_probe, {'a': a_slider, 'b': b_slider})
    display(VBox([a_slider, b_slider, HBox([Label('Preset:'), p_button]), plot_screen, readout]))
else:
    print('ipywidgets is not available here. Use the static cell below instead.')

*Figure description.* Left: a grey mesh of the surface. A solid purple curve is the $x$-slice and a dashed purple curve is the $y$-slice. Two green lines touch the surface at the amber probe: the solid one along the $x$-slice and the dashed one along the $y$-slice.
Middle: the $x$-slice alone, drawn as an ordinary graph of $z$ against $x$, with its tangent line at the probe. The title prints $f_x(a,b)$.
Right: the $y$-slice alone, drawn against $y$, with its tangent line. The title prints $f_y(a,b)$.

In [ ]:
# Static version (no widgets needed). Change a_static and b_static and run again.
a_static, b_static = -1.5, -1.0
plot_probe(a_static, b_static)

*Figure description.* The same three panels as above, drawn once at the point stored in `a_static` and `b_static`.

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🧠 Explain.</strong> Put the probe at $P$. Read the two slopes. Which slice is steeper at $P$?
<br><br>Now explain <b>in one sentence</b>, using $f_x=-x$ and $f_y=-2y$: why does the slope of the $x$-slice stay the same when you move only $b$?
<br><br>Then find the point where both slices are level. What is special about the surface there? Compare with Example 3 on the page.

</div>

Write your sentence first. Then run the next cell to compare the readout with your Section 1 prediction.

In [ ]:
# Compare with your Section 1 prediction: the probe at P = (1/2, 1).
plot_probe(float(P[0]), float(P[1]))

*Figure description.* The same three panels, drawn at =\left(\tfrac12,1\right)$. Read the tilt of each green line. Then compare it with the two guesses you logged in Section 1.

---
## 5 · CHALLENGE · Two slopes are not the whole surface

🎯 **STAGE 4/7 · CHALLENGE** ▰▰▰▰▱▱▱

Example 3 on the page is the surface $f(x,y)=1-(x-1)^2-(y-2)^2$ at $(1,2,1)$. Both slopes are $0$ there. The page calls this point the **peak**.

The page's *Common errors* list names a second surface: the saddle $s(x,y)=x^2-y^2$. Its partials at $(0,0)$ are also both $0$.
The cell below checks both pairs of numbers. Then it draws the two slices of each surface.

In [ ]:
peak_expr = 1 - (x - 1)**2 - (y - 2)**2
saddle_expr = x**2 - y**2
for name, F, pt in [('Example 3 at (1, 2)', peak_expr, (1, 2)), ('saddle x^2 - y^2 at (0, 0)', saddle_expr, (0, 0))]:
    fx_pt = partial_limit(F, x, pt)
    fy_pt = partial_limit(F, y, pt)
    print(f'{name:<28} f_x = {fx_pt},  f_y = {fy_pt}')

t = np.linspace(-1.6, 1.6, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
# Left: the two slices of Example 3 through (1, 2), shifted so that the point sits at offset 0.
ax1.plot(t, 1 - t**2, color=NEGATIVE, lw=2.4, label='red: x-slice z = f(1 + t, 2), curves down')
ax1.plot(t, 1 - t**2, color=SLICE, lw=2.0, ls='--', label='purple dashed: y-slice z = f(1, 2 + t), curves down')
ax1.axhline(1, color=TANGENT, lw=1.8, ls=':', label='green dotted: both tangent lines (slope 0)')
ax1.plot([0], [1], 'o', color=PROBE, ms=9, label='amber dot: the point (1, 2, 1)')
ax1.set_title('Example 3: both slices curve down (a peak)')
ax1.set_xlabel('offset t from the point along the slice')
ax1.set_ylabel('z')
ax1.grid(alpha=0.3)
ax1.legend(loc='lower center', fontsize=8.5)
# Right: the two slices of the saddle through (0, 0).
ax2.plot(t, t**2, color=PRIMARY, lw=2.4, label='blue: x-slice z = s(t, 0) = t², curves up')
ax2.plot(t, -t**2, color=NEGATIVE, lw=2.4, ls='--', label='red dashed: y-slice z = s(0, t) = −t², curves down')
ax2.axhline(0, color=TANGENT, lw=1.8, ls=':', label='green dotted: both tangent lines (slope 0)')
ax2.plot([0], [0], 'o', color=PROBE, ms=9, label='amber dot: the point (0, 0, 0)')
ax2.set_title('Saddle x² − y²: one slice up, one slice down')
ax2.set_xlabel('offset t from the point along the slice')
ax2.grid(alpha=0.3)
ax2.legend(loc='lower center', fontsize=8.5)
ax2.set_ylim(-2.8, 2.8)
hud_frame(fig, 'SEC 5 · TWO SLOPES, TWO SURFACES', footer='SLICE PROBE · Example 3 and the saddle x² − y²')
plt.show()

*Figure description.* Two panels with the same vertical scale. The horizontal axis is the offset $t$ from the point along each slice.
Left, Example 3: the $x$-slice (red) and the $y$-slice (purple dashed) are the same downward parabola $1-t^2$, so they lie on top of each other. Both have a level tangent (green dotted) at the amber point.
Right, the saddle: the $x$-slice $t^2$ (blue) curves up and the $y$-slice $-t^2$ (red dashed) curves down. Both still have a level tangent at the amber point.

<div style="background:#FFFBEB;border-left:6px solid #D97706;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🎯 Challenge question.</strong> Both points report the same two numbers: $f_x=0$ and $f_y=0$. Yet one point is a peak and the other is a saddle. What do the two slices show that the two numbers do not? Answer in two or three sentences.

</div>

<details>
<summary>Hint (open after you try)</summary>

A slope tells you the tilt of the tangent line. It says nothing about whether the slice bends up or down. The page's *Common errors* list uses this saddle. Its *Try this* box (in *Interactive sandbox*) measures the bending with second partials: for $x^2-y^2$, $f_{xx}=2>0$ and $f_{yy}=-2<0$. Lecture 20 returns to this idea.

</details>

---
## 6 · CHALLENGE · Higher-order partials and Clairaut's theorem

🎯 **STAGE 4/7 · CHALLENGE** ▰▰▰▰▱▱▱

The first partials are again functions of $x$ and $y$. So we can differentiate them again.
The page writes $f_{xy}=(f_x)_y$: first $x$, then $y$. In sympy, `sp.diff(f, x, y)` means the same thing.

**Theorem (Clairaut).** If both $f_{xy}$ and $f_{yx}$ are continuous on a neighborhood of $(a,b)$, then $f_{xy}(a,b)=f_{yx}(a,b)$.

**Page Example 9.** Let $f(x,y)=3xy^2-2y+5x^2y^2$. Find the second partials and $f_{xy}(-1,2)$.
The cell below computes $f_{xy}$ and $f_{yx}$ by their two different routes.

In [ ]:
f9 = 3*x*y**2 - 2*y + 5*x**2*y**2
f9_x, f9_y = sp.diff(f9, x), sp.diff(f9, y)
f9_xx, f9_yy = sp.diff(f9_x, x), sp.diff(f9_y, y)
f9_xy = sp.diff(f9_x, y)      # first x, then y
f9_yx = sp.diff(f9_y, x)      # first y, then x
show_math(r'f_x = ' + sp.latex(f9_x) + r',\qquad f_y = ' + sp.latex(sp.expand(f9_y)))
show_math(r'f_{xx} = ' + sp.latex(f9_xx) + r',\qquad f_{yy} = ' + sp.latex(sp.expand(f9_yy)))
show_math(r'f_{xy} = ' + sp.latex(f9_xy) + r',\qquad f_{yx} = ' + sp.latex(f9_yx))
print('f_xy and f_yx agree everywhere:', sp.simplify(f9_xy - f9_yx) == 0)
show_math(r'f_{xy}(-1,2) = ' + sp.latex(f9_xy.subs({x: -1, y: 2})))

**Step 1 · Clairaut's hypothesis:** $f_{xy}$ and $f_{yx}$ continuous near $(-1,2)$.
- Observe that both mixed partials are polynomials.
- A polynomial is continuous everywhere.
- So the hypothesis holds on a neighborhood of every point.

**Step 2 · Meaning of the mixed partial:** $f_{xy}=(f_x)_y$.
- $f_{xy}$ is the rate at which the $x$-slope changes as we move in the $y$-direction.

**Step 3 · The $x$-slope along a line:** $f_x(-1,y)=-7y^2$.
- Freeze $x=-1$ in $f_x$. This gives the $x$-slope on the line $x=-1$.
- Its slope at $y=2$ should be $f_{xy}(-1,2)$.

**Answer.** Clairaut applies at every point, so $f_{xy}=f_{yx}$. The value $f_{xy}(-1,2)$ is the slope of the curve $f_x(-1,y)$ at $y=2$. The figure below checks this.

In [ ]:
slope_line = sp.expand(f9_x.subs(x, -1))          # the x-slope along the line x = -1
slope_line_num = sp.lambdify(y, slope_line, 'numpy')
rate = float(sp.diff(slope_line, y).subs(y, 2))
show_math(r'f_x(-1,y) = ' + sp.latex(slope_line) + r',\qquad \frac{d}{dy}f_x(-1,y)\Big|_{y=2} = ' + fmt(rate, 0))

yy = np.linspace(0.5, 3.0, 200)
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(yy, slope_line_num(yy), color=PRIMARY, lw=2.4, label=r'blue: the $x$-slope $f_x(-1,y)$ along the line $x=-1$')
ss = np.linspace(-0.6, 0.6, 2)
ax.plot(2 + ss, float(slope_line_num(2)) + rate*ss, color=TANGENT, lw=2.4,
        label=r'green: tangent line at $y=2$, slope $f_{xy}(-1,2)$')
ax.plot([2], [float(slope_line_num(2))], 'o', color=PROBE, ms=9, label='amber dot: y = 2')
ax.set_xlabel('y  (with x = −1 fixed)')
ax.set_ylabel(r'$f_x(-1,y)$')
ax.set_title(r'Example 9: $f_{xy}(-1,2)$ is the rate of change of the $x$-slope in the $y$-direction')
ax.grid(alpha=0.3)
ax.legend(loc='lower left', fontsize=9)
hud_frame(fig, 'SEC 6 · MIXED PARTIAL', footer='SLICE PROBE · Example 9 · f = 3xy² − 2y + 5x²y²')
plt.show()

*Figure description.* The blue curve is the $x$-slope $f_x(-1,y)=-7y^2$ plotted against $y$. It falls faster and faster as $y$ grows. The green line touches it at the amber dot $y=2$. Its slope is the mixed partial $f_{xy}(-1,2)$.

<div style="background:#FFFBEB;border-left:6px solid #D97706;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🎯 Link back to the probe.</strong> In Section 4 the slope of the $x$-slice did not change when you moved only $b$. Which mixed partial of Example 2 does this observation measure, and what is its value? Check your answer with one line of sympy.

</div>

In [ ]:
# ✏️ Optional: one line of sympy for the mixed partial of Example 2 (f_expr), first x then y.
my_mixed = None    # for example sp.diff(f_expr, ?, ?)
print('Not computed yet.' if my_mixed is None else f'Your mixed partial: {my_mixed}')

---
## 7 · TRANSFER

🚀 **STAGE 5/7 · TRANSFER** ▰▰▰▰▰▱▱

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🚀 Transfer, without the probe.</strong> Page Example 6: the area of a parallelogram with adjacent sides $a$ and $b$ and included angle $\theta$ is $A=ab\sin\theta$. Now there are three variables and no picture.

1. To find $\partial A/\partial a$, which quantities are frozen? Find $\partial A/\partial a$ at $a=10$, $b=20$, $\theta=\pi/6$.
2. Find $\partial A/\partial\theta$ at the same values. Give the exact value and a decimal.

</div>

**Transfer note.** The page's hook uses the same operation. Delta $=\partial V/\partial S$ is the rate of change of an option price $V$ when only the stock price $S$ moves and the other variables are held fixed.

In [ ]:
# ✏️ Optional check. Type your two formulas with the symbols below, then run.
a_s, b_s, th = sp.symbols('a b theta', positive=True)
A_expr = a_s * b_s * sp.sin(th)
my_dA_da = None        # your formula for dA/da, for example b_s*sp.sin(th)
my_dA_dtheta = None    # your formula for dA/dtheta

for name, mine, var in [('dA/da', my_dA_da, a_s), ('dA/dtheta', my_dA_dtheta, th)]:
    if mine is None:
        print(f'{name}: fill in your formula first.')
    else:
        same = sp.simplify(mine - sp.diff(A_expr, var)) == 0
        print(f'{name}: ' + ('matches sympy.' if same else 'does not match yet. Check which quantities you froze.'))

<details>
<summary>Check your answer</summary>

**Step 1 · Partial in $a$:** $\partial A/\partial a=b\sin\theta$.
- Freeze $b$ and $\theta$. Differentiate $A=ab\sin\theta$ in $a$.
- Substitute $b=20$, $\theta=\pi/6$: $\partial A/\partial a=20\cdot\tfrac12=10$.

**Step 2 · Partial in $\theta$:** $\partial A/\partial\theta=ab\cos\theta$.
- Freeze $a$ and $b$. Differentiate $A=ab\sin\theta$ in $\theta$.
- Substitute $a=10$, $b=20$, $\theta=\pi/6$: $\partial A/\partial\theta=200\cdot\tfrac{\sqrt3}{2}=100\sqrt3\approx173.2$.

**Answer.** $\partial A/\partial a=10$ and $\partial A/\partial\theta=100\sqrt3\approx173.2$. These are the values on the page.

</details>

---
## 8 · CREATE · Mission log

📝 **STAGE 6/7 · CREATE** ▰▰▰▰▰▰▱

Fill in the log with your own readings from Sections 2–4. Numbers may be decimals or exact strings such as `"-1/2"`.
Then write one sentence in your own words.

In [ ]:
# ✏️ CREATE. Fill in your mission log, then run this cell and the next one.
mission_log = {
    'f_x_P': None,            # the slope of the x-slice at P = (1/2, 1)
    'f_y_P': None,            # the slope of the y-slice at P
    'steeper_slice': None,    # 'x' or 'y'
    'level_point': [None, None],   # the point (a, b) where both slices are level
    'one_sentence': '',       # what does f_x(P) measure on the surface? Use the word "slice".
}

In [ ]:
# Check the mission log. The card shows only "correct" or "try again", never the expected values.
level_sol = sp.solve([fx_expr, fy_expr], [x, y], dict=True)   # both slopes zero
level_target = [level_sol[0][x], level_sol[0][y]]
steeper_target = 'x' if abs(float(fx_P_rule)) > abs(float(fy_P_rule)) else 'y'
checks = [('f_x_P', check_answer(mission_log['f_x_P'], fx_P_rule)),
          ('f_y_P', check_answer(mission_log['f_y_P'], fy_P_rule)),
          ('steeper_slice', check_answer(mission_log['steeper_slice'], steeper_target)),
          ('level_point', check_answer(mission_log['level_point'], level_target, tol=0.03))]
rows = [(key, '✅', 'correct') if ok else (key, '🔁', 'try again') for key, ok in checks]
sentence = mission_log['one_sentence'].strip()
rows.append(('one_sentence', '✅', 'logged') if sentence else ('one_sentence', '✏️', 'write one sentence'))

n_ok = sum(ok for _, ok in checks)
mission_log_ok = n_ok == len(checks) and bool(sentence)
summary = f'{n_ok} of {len(checks)} correct.'
summary += ' Sentence logged.' if sentence else ' Sentence not written yet.'
if mission_log_ok:
    summary += ' Mission log complete. Next: the debrief.'
check_card('MISSION LOG · CHECK', rows, summary)
if mission_log_ok and globals().get('quiz_ok', False):
    mission_complete_card()

---
## 9 · DEBRIEF · ⚠️ Common errors: check yourself against these

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#FEF2F2;border-left:6px solid #DC2626;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>⚠️ Common errors.</strong>

- **Treating the frozen variable as if it disappeared.** In $x^2e^{xy}$, holding $y$ fixed does *not* make $e^{xy}$ a constant. It still contains $x$. Example 4(b) needs the product rule *and* the chain rule.
- **Moving both coordinates in the difference quotient.** In the definition of $f_x(a,b)$ only $x$ moves: $f(a+h,b)$. The engine of Section 2 enforces this.
- **Reading the mixed-partial notation backwards.** $f_{xy}$ means "$x$ first, then $y$", and so does $\dfrac{\partial^2 f}{\partial y\,\partial x}$. The subscript form reads left to right. The $\partial$ form reads right to left.
- **Using the rules at a seam.** For a piecewise $f$, differentiating the top formula and substituting the special point does not compute the partial there. Go back to the definition.
- **Concluding a surface is flat because both partials vanish.** They report only two directions. The saddle $x^2-y^2$ has $f_x(0,0)=f_y(0,0)=0$ and is not flat (Section 5).
- **Quoting Clairaut without checking its hypothesis.** The theorem requires the mixed partials to be **continuous**. Without that check the conclusion can fail.

</div>

## 10 · DEBRIEF · Retrieval check: mini-quiz

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

Fill in `student_answers` from memory, then run the cell. Numbers may be decimals or exact strings such as `"-1/2"`.
The check card shows only **correct** or **try again**. After your second try you may set `reveal = True` to see the key.

In [ ]:
# ✏️ Mini-quiz. All items come from the Lecture 15 page.
student_answers = {
    'f_x_from_definition': None,            # f_x(1, 2) for f = x^2 + 3xy (page, Using the definition directly)
    'example1_f_y_at_2_m4': None,           # f_y(2, -4) for f = x^2 + y^2 - 4 (Example 1)
    'example3_slopes': None,                # [f_x(1, 2), f_y(1, 2)] for f = 1 - (x-1)^2 - (y-2)^2 (Example 3)
    'example9_f_xy_at_m1_2': None,          # f_xy(-1, 2) for f = 3xy^2 - 2y + 5x^2y^2 (Example 9)
    'both_partials_zero_means_flat': None,  # True or False
}
reveal = False   # set to True only after your second try

_key = json.loads(base64.b64decode('eyJmX3hfZnJvbV9kZWZpbml0aW9uIjogOCwgImV4YW1wbGUxX2ZfeV9hdF8yX200IjogLTgsICJleGFtcGxlM19zbG9wZXMiOiBbMCwgMF0sICJleGFtcGxlOV9mX3h5X2F0X20xXzIiOiAtMjgsICJib3RoX3BhcnRpYWxzX3plcm9fbWVhbnNfZmxhdCI6IGZhbHNlfQ=='))
if any(v is not None for v in student_answers.values()):
    quiz_attempts = globals().get('quiz_attempts', 0) + 1
else:
    quiz_attempts = globals().get('quiz_attempts', 0)

n_correct = 0
rows = []
for name, target in _key.items():
    ok = check_answer(student_answers[name], target)
    n_correct += ok
    rows.append((name, '✅', 'correct') if ok else (name, '🔁', 'try again'))
quiz_ok = n_correct == len(_key)
check_card('RETRIEVAL · CHECK', rows, f'{n_correct} of {len(_key)} correct · attempt {quiz_attempts}')

if reveal and quiz_attempts < 2:
    print('Try at least twice before you reveal the key.')
elif reveal:
    print('Key:', _key)
elif quiz_ok and globals().get('mission_log_ok', False):
    mission_complete_card()
elif quiz_ok:
    print('All correct. Finish the mission log in Stage 6 to complete the mission.')

## 11 · DEBRIEF · 🤖 AI Learning Companion

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:6px 14px;border-radius:8px;color:#1E293B;">
<strong>🤖 Student-facing AI contract — paste this at the start of every AI session</strong>
<br>1. Ask me what I have already tried before you explain anything.
<br>2. Do not give a complete solution until I have made a prediction or proposed a next step.
<br>3. Point to the first gap in my reasoning, not every later consequence.
<br>4. Make me verify the result by units, sign, substitution, a graph, or a limiting case.
<br>5. End with one transfer question that changes the context.
</div>

<div style="background:#F8FAFC;border-left:6px solid #94A3B8;padding:6px 14px;border-radius:8px;margin-top:6px;color:#1E293B;">
<strong>🔎 Verification rule.</strong> Treat an AI answer as a conjecture. Check it against the definitions, theorems and figures on this page. Bring any disagreement to class. An AI answer never overrides a checked definition or proof. Neither does an illustration.
</div>

**Prompt for this lab: Error Hunter.** Paste your own partial-derivative solution in place of `[paste my work]`.

```
Here is my solution to a partial-derivative or higher-order-partial problem:
[paste my work]
Find the FIRST invalid step. Do not fix it. Instead:
(1) quote that step, (2) name the underlying misconception in one sentence,
(3) give me a two-line counterexample or sanity check that exposes it,
(4) ask me to repair the step myself.
If my solution is correct, instead give me a harder variant that breaks the
method I used, and ask me why it breaks.
```

## 12 · DEBRIEF · 🔒 Independence check

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:8px 16px;border-radius:8px;color:#1E293B;">
<strong>🔒 Independence rule.</strong> <i>No AI, no hints, new numbers. This shows that you can do it alone.</i>
</div>

Let $q(x,y)=x^3-2xy^2+4y$.

1. Find $q_x$ and $q_y$ by the rule.
2. Find $q_x(2,-1)$ **from the limit definition**. Then check it with the rule.
3. At $(2,-1)$, which slice is steeper: the $x$-slice or the $y$-slice?
4. Find $q_{xy}$ and $q_{yx}$. Do they agree? Say why Clairaut's theorem was entitled to apply.

Do this on paper, without AI and without the worked examples. It is not checked here. It is practice for HW5 and HW6.

---
## 13 · OPTIONAL · Review examples from the lecture page

**OPTIONAL · outside the seven stages.** Skip this part if you are short of time. The mission ends with the prediction for Lecture 16 below.

These are worked examples from the Lecture 15 page, solved with sympy. Try each one on paper before you run its cell.

### 13.1 · Example 1 · Finding partial derivatives

Let $f(x,y)=x^2+y^2-4$. Compute $f_x$ and $f_y$, and evaluate both at $(2,-4)$.

In [ ]:
f1 = x**2 + y**2 - 4
show_math(r'f_x = ' + sp.latex(sp.diff(f1, x)) + r',\qquad f_y = ' + sp.latex(sp.diff(f1, y)))
show_math(r'f_x(2,-4) = ' + sp.latex(sp.diff(f1, x).subs({x: 2, y: -4}))
          + r',\qquad f_y(2,-4) = ' + sp.latex(sp.diff(f1, y).subs({x: 2, y: -4})))

### 13.2 · Example 8 · A function of three variables

Let $f(x,y,z)=e^{xy}\ln z$. Hold two variables fixed at a time.

In [ ]:
z = sp.symbols('z', positive=True)
f8 = sp.exp(x*y) * sp.log(z)
for var in (x, y, z):
    show_math(r'\frac{\partial f}{\partial ' + sp.latex(var) + '} = ' + sp.latex(sp.diff(f8, var)))

### 13.3 · A first look at partial differential equations (worked example)

The page shows that $z=e^x\sin y$ satisfies Laplace's equation $z_{xx}+z_{yy}=0$. The cell repeats the page's check.

In [ ]:
zL = sp.exp(x) * sp.sin(y)
show_math(r'z_{xx} = ' + sp.latex(sp.diff(zL, x, 2)) + r',\qquad z_{yy} = ' + sp.latex(sp.diff(zL, y, 2)))
show_math(r'z_{xx} + z_{yy} = ' + sp.latex(sp.simplify(sp.diff(zL, x, 2) + sp.diff(zL, y, 2))))

### 13.4 · Example 10 · Mixed partials of three variables

Let $f(x,y,z)=y\,e^x+x\ln z$ with $z>0$. Verify that $f_{xz}=f_{zx}$ and $f_{xzz}=f_{zxz}=f_{zzx}$.

In [ ]:
f10 = y*sp.exp(x) + x*sp.log(z)
show_math(r'f_{xz} = ' + sp.latex(sp.diff(f10, x, z)) + r',\qquad f_{zx} = ' + sp.latex(sp.diff(f10, z, x)))
show_math(r'f_{xzz} = ' + sp.latex(sp.diff(f10, x, z, z)) + r',\quad f_{zxz} = ' + sp.latex(sp.diff(f10, z, x, z))
          + r',\quad f_{zzx} = ' + sp.latex(sp.diff(f10, z, z, x)))

*Note.* All partials of $f$ are continuous on the domain $z>0$. So Clairaut's theorem and its extension in the page's *Remark* apply there.

---
## 14 · DEBRIEF · Prediction for Lecture 16

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

The mission is complete when the Stage 6 log and the retrieval check both show ✅ for every item.
Your last task is one prediction.

<div style="background:#0A0E14;border:1px solid #22D3EE;border-radius:10px;padding:10px 16px;color:#E2E8F0;">
<span style="color:#22D3EE;font-family:Menlo,Consolas,monospace;letter-spacing:1px;font-size:12px;">MISSION 15 · FINAL PROMPT · PREDICTION FOR LECTURE 16</span><br>
Both partials exist at a point. Is the tangent plane built from them always a good approximation?
<br>Write your guess in one sentence. Lecture 16 will test it.
</div>

---
<div style="background:#0F1A2B;border-top:2px solid #22D3EE;color:#94A3B8;padding:8px 16px;font-family:Menlo,Consolas,monospace;font-size:12px;letter-spacing:1px;">
END OF LAB 15 · SLICE PROBE · © Tan H. Cao, for AMS 261 students
</div>